# Mission 04: persistent_memory (Persistent Working Memory L1) - 해답 노트북

이 노트북은 두 번째 미션의 완성된 솔루션 코드와 설명입니다.

In [ ]:
# 1. 필요한 라이브러리 및 환경 로드
import sys
import os
from dotenv import load_dotenv

while not os.path.exists("app") and os.getcwd() != "/":
    os.chdir("..")
# 루트 폴더 기준의 경로 등록
sys.path.append(os.path.abspath("src"))
sys.path.append(os.path.abspath("."))  # sys.path.append("app") 대신 "." 등록이 파이썬 패키지 경로 탐색에 안전합니다.
load_dotenv(override=True)

from app.utils.llm import get_llm
from langchain.agents import create_agent
from langgraph.checkpoint.sqlite import SqliteSaver
from langchain_core.messages import HumanMessage

### [미션 1] SQLite 연결 및 SqliteSaver 객체 생성하기

데이터베이스 경로 `app/database/checkpoints.db`에 sqlite3 연결을 설정하고 `SqliteSaver`를 빌드하는 코드입니다.

In [ ]:
import sqlite3

db_dir = "app/database"
os.makedirs(db_dir, exist_ok=True)
db_path = os.path.join(db_dir, "checkpoints.db")

# sqlite3.connect를 이용하여 db_path 연결 객체 생성
conn = sqlite3.connect(db_path, check_same_thread=False)

# SqliteSaver 연결 래핑 객체 빌드
memory = SqliteSaver(conn)

if memory is not None:
    print("✅ SqliteSaver 생성 성공!")
else:
    print("❌ SqliteSaver 생성 실패!")

### [미션 2] 에이전트에 영속 체커바인더 주입 및 1차 대화 테스트

생성한 `memory`를 체크포인터로 넘겨 에이전트를 빌드하고, 사용자 이름을 기억하도록 만듭니다.

In [ ]:
llm = get_llm(model_name="gemini-3.5-flash", temperature=0.0)
from app.prompts import CHATBOT_SYSTEM_PROMPT
from app.utils.context import AgentContext

# create_agent에 checkpointer로 memory 객체를 주입
agent = create_agent(
    model=llm,
    tools=[],
    system_prompt=CHATBOT_SYSTEM_PROMPT,
    checkpointer=memory,
    context_schema=AgentContext
)

# 세션 아이디 및 테스트 질문 설정
thread_id = "persistence_test_session_001"
config = {"configurable": {"thread_id": thread_id}}

if agent:
    # 첫 대화 진행
    res1 = agent.invoke(
        {"messages": [HumanMessage(content="내 이름은 김영희이고, 취미는 테니스 치는 거야.")]},
        config=config
    )
    print("답변1:", res1["messages"][-1].content)
else:
    print("에이전트가 완성되지 않았습니다.")

### [미션 3] 영속성 최종 검증

새로운 SQLite 세션을 맺고 완전히 다른 에이전트 인스턴스를 빌드한 후 동일한 `thread_id`로 질문하여 영속 상태를 검증합니다.

In [ ]:
# 기존 커넥션을 닫고 새 커넥션으로 에이전트를 완전 새로 인스턴스화합니다.
if conn:
    conn.close()

new_conn = sqlite3.connect(db_path, check_same_thread=False)
new_memory = SqliteSaver(new_conn)

# 새 에이전트 재생성
new_agent = create_agent(
    model=llm,
    tools=[],
    system_prompt=CHATBOT_SYSTEM_PROMPT,
    checkpointer=new_memory,
    context_schema=AgentContext
)

# 동일한 thread_id로 기억력 테스트
res2 = new_agent.invoke(
    {"messages": [HumanMessage(content="내 이름이 뭔지, 그리고 내 취미가 뭔지 다시 기억나?")]},
    config=config
)
print("답변2:", res2["messages"][-1].content)

new_conn.close()